# MaS-TransUNet — Colab Training Notebook

This notebook trains and evaluates the MaS-TransUNet model for medical image segmentation.
Run each cell in order. Read the markdown instructions before each cell carefully.

---
## 1. Mount Google Drive

This mounts your Google Drive to `/content/drive` so datasets and checkpoints persist across sessions.

**Before proceeding:** Make sure you are logged into the Google account that contains your datasets under `My Drive/datasets/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Install Dependencies

Installs all required packages and verifies CUDA is available.

**Before proceeding:** Check that the output shows a GPU name (e.g. `Tesla T4`). If it says CUDA is not available, go to `Runtime > Change runtime type` and select **GPU**.

In [ ]:
!pip install -q torch torchvision timm==0.9.12 einops albumentations \
    opencv-python-headless scikit-image scikit-learn monai matplotlib \
    seaborn pandas tqdm tensorboard scipy Pillow

import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is NOT available. Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

---
## 3. Clone Repository

Clones the MaS-TransUNet repository from GitHub and enters the project directory.

**Repository URL:** `https://github.com/Surfing-Ninja/TransUNet.git`
On reconnection, `git pull` ensures you have the latest code.

In [ ]:
import os

REPO_URL = "https://github.com/Surfing-Ninja/TransUNet.git"
REPO_DIR = "/content/TransUNet"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git pull
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Auto-unzip and organize datasets from Google Drive
# Run this once after mounting Drive (Cell 1).

import os
import re
import glob
import zipfile
import shutil
import random
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive"
DATASETS_ROOT = os.path.join(DRIVE_ROOT, "datasets")
os.makedirs(DATASETS_ROOT, exist_ok=True)

EXPECTED = ["tcga_lgg", "dsb2018", "kvasir_seg", "isic2018", "covid_ct"]

# Search for zip files (supports your archive*.zip uploads)
zip_patterns = [
    os.path.join(DRIVE_ROOT, "archive*.zip"),
    os.path.join(DRIVE_ROOT, "datasets", "archive*.zip"),
    os.path.join(DRIVE_ROOT, "dataset_zips", "archive*.zip"),
    os.path.join(DRIVE_ROOT, "*.zip"),
]

zip_paths = []
for pattern in zip_patterns:
    zip_paths.extend(glob.glob(pattern))
zip_paths = sorted(set(zip_paths))

if not zip_paths:
    raise FileNotFoundError(
        "No zip files found in MyDrive. Upload archive*.zip files to MyDrive and re-run this cell."
    )

print("Found zip files:")
for p in zip_paths:
    print(" -", p)


def infer_dataset_name(zip_path, namelist_lower, used_names):
    name = os.path.basename(zip_path).lower()
    content = " ".join(namelist_lower[:200])  # inspect head for speed
    text = f"{name} {content}"

    rules = [
        ("tcga_lgg", ["tcga", "lgg"]),
        ("dsb2018", ["dsb", "bowl", "nuclei", "2018"]),
        ("kvasir_seg", ["kvasir", "polyp"]),
        ("isic2018", ["isic", "lesion", "skin"]),
        ("covid_ct", ["covid", "ct"]),
    ]

    for ds_name, keys in rules:
        if all(k in text for k in keys[:1]) and ds_name not in used_names:
            return ds_name

    # fallback: first unused expected name
    for ds_name in EXPECTED:
        if ds_name not in used_names:
            return ds_name

    return None


def find_first_dir_with_any(root_dir, candidates):
    for c in candidates:
        p = os.path.join(root_dir, c)
        if os.path.isdir(p):
            return p

    for cur, dirs, _ in os.walk(root_dir):
        dset = {d.lower() for d in dirs}
        for c in candidates:
            if c.lower() in dset:
                return os.path.join(cur, c)
    return None


def list_image_files(folder):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = []
    for p in Path(folder).rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            files.append(str(p))
    return sorted(files)


def stem_key(path):
    base = os.path.splitext(os.path.basename(path))[0].lower()
    base = re.sub(r"(_mask|_masks|_seg|_label|_labels|mask|seg)$", "", base)
    return base


def ensure_structure(dataset_dir, seed=42, test_ratio=0.2):
    train_images = os.path.join(dataset_dir, "train", "images")
    train_masks = os.path.join(dataset_dir, "train", "masks")
    test_images = os.path.join(dataset_dir, "test", "images")
    test_masks = os.path.join(dataset_dir, "test", "masks")

    # If already correct, just ensure edge folders
    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)
        return "already-structured"

    # Common alternative: val instead of test
    val_images = os.path.join(dataset_dir, "val", "images")
    val_masks = os.path.join(dataset_dir, "val", "masks")
    valid_images = os.path.join(dataset_dir, "valid", "images")
    valid_masks = os.path.join(dataset_dir, "valid", "masks")

    if os.path.isdir(train_images) and os.path.isdir(train_masks):
        if os.path.isdir(val_images) and os.path.isdir(val_masks):
            os.makedirs(os.path.join(dataset_dir, "test"), exist_ok=True)
            if os.path.exists(test_images):
                shutil.rmtree(test_images)
            if os.path.exists(test_masks):
                shutil.rmtree(test_masks)
            shutil.move(val_images, test_images)
            shutil.move(val_masks, test_masks)
            val_root = os.path.join(dataset_dir, "val")
            if os.path.isdir(val_root):
                shutil.rmtree(val_root, ignore_errors=True)
        elif os.path.isdir(valid_images) and os.path.isdir(valid_masks):
            os.makedirs(os.path.join(dataset_dir, "test"), exist_ok=True)
            if os.path.exists(test_images):
                shutil.rmtree(test_images)
            if os.path.exists(test_masks):
                shutil.rmtree(test_masks)
            shutil.move(valid_images, test_images)
            shutil.move(valid_masks, test_masks)
            valid_root = os.path.join(dataset_dir, "valid")
            if os.path.isdir(valid_root):
                shutil.rmtree(valid_root, ignore_errors=True)

    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)
        return "converted-val-to-test"

    # Flat layout fallback: images/ + masks/ at any depth -> split train/test
    images_dir = find_first_dir_with_any(dataset_dir, ["images", "imgs", "image"])
    masks_dir = find_first_dir_with_any(dataset_dir, ["masks", "mask", "labels", "label", "ground_truth", "gt"])

    if not images_dir or not masks_dir:
        raise RuntimeError(f"Could not find image/mask folders in {dataset_dir}")

    img_files = list_image_files(images_dir)
    mask_files = list_image_files(masks_dir)

    mask_map = {stem_key(m): m for m in mask_files}
    pairs = []
    for img in img_files:
        k = stem_key(img)
        if k in mask_map:
            pairs.append((img, mask_map[k]))

    if len(pairs) == 0:
        raise RuntimeError(f"No matching image/mask pairs found in {dataset_dir}")

    random.seed(seed)
    random.shuffle(pairs)

    n_total = len(pairs)
    n_test = max(1, int(n_total * test_ratio)) if n_total > 1 else 0
    n_train = n_total - n_test

    train_pairs = pairs[:n_train]
    test_pairs = pairs[n_train:]

    for folder in [train_images, train_masks, test_images, test_masks]:
        os.makedirs(folder, exist_ok=True)

    def copy_pairs(pairs_list, img_out, mask_out):
        for i, (img, msk) in enumerate(pairs_list):
            ext_i = os.path.splitext(img)[1].lower() or ".png"
            ext_m = os.path.splitext(msk)[1].lower() or ".png"
            name = f"{i:05d}"
            shutil.copy2(img, os.path.join(img_out, name + ext_i))
            shutil.copy2(msk, os.path.join(mask_out, name + ext_m))

    copy_pairs(train_pairs, train_images, train_masks)
    copy_pairs(test_pairs, test_images, test_masks)

    os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
    os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)

    return f"split-flat-layout ({len(train_pairs)} train / {len(test_pairs)} test)"


used = set()
summary = []

for zp in zip_paths:
    with zipfile.ZipFile(zp, "r") as zf:
        names = [n.lower() for n in zf.namelist() if n and not n.endswith("/")]

    ds_name = infer_dataset_name(zp, names, used)
    if ds_name is None:
        print(f"[SKIP] Could not assign dataset name for {zp}")
        continue

    used.add(ds_name)
    ds_dir = os.path.join(DATASETS_ROOT, ds_name)
    tmp_extract = f"/content/_tmp_extract_{ds_name}"

    if os.path.isdir(tmp_extract):
        shutil.rmtree(tmp_extract)
    os.makedirs(tmp_extract, exist_ok=True)

    print(f"\n[UNZIP] {os.path.basename(zp)} -> {ds_name}")
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(tmp_extract)

    # Find the best candidate root that contains image/mask content
    candidate_root = tmp_extract
    for root, dirs, files in os.walk(tmp_extract):
        d = {x.lower() for x in dirs}
        if {"train", "test"}.issubset(d):
            candidate_root = root
            break
        if any(k in d for k in ["images", "masks", "labels", "ground_truth", "gt"]):
            candidate_root = root

    # Merge extracted content into final dataset dir
    os.makedirs(ds_dir, exist_ok=True)
    for item in os.listdir(candidate_root):
        src = os.path.join(candidate_root, item)
        dst = os.path.join(ds_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.move(src, dst)
        else:
            # keep stray files but do not overwrite
            if not os.path.exists(dst):
                shutil.move(src, dst)

    result = ensure_structure(ds_dir)
    summary.append((ds_name, result))

    shutil.rmtree(tmp_extract, ignore_errors=True)

print("\n=== Dataset Preparation Summary ===")
for ds_name, result in summary:
    train_images = os.path.join(DATASETS_ROOT, ds_name, "train", "images")
    test_images = os.path.join(DATASETS_ROOT, ds_name, "test", "images")
    n_train = len(list(Path(train_images).glob("**/*"))) if os.path.isdir(train_images) else 0
    n_test = len(list(Path(test_images).glob("**/*"))) if os.path.isdir(test_images) else 0
    print(f"- {ds_name:10s} | {result:30s} | train files: {n_train:5d} | test files: {n_test:5d}")

print("\nDone. Next: run the Set Colab Mode cell to verify all datasets show Found.")

---
## 4. Set Colab Mode

Switches all config paths to Google Drive locations and sets the device to CUDA.

**Before proceeding:** Verify each dataset path prints **Found**. If any path prints **Missing**, upload the dataset to the corresponding Drive folder before training on that dataset.

In [ ]:
import os
from config import CFG

# Switch to Colab paths
CFG.is_colab = True
CFG.__post_init__()  # Re-derive paths with Colab flag
CFG.device = "cuda"

print(f"Base data dir:   {CFG.base_data_dir}")
print(f"Checkpoint dir:  {CFG.checkpoint_dir}")
print(f"Device:          {CFG.device}")
print()

for name, paths in CFG.dataset_paths.items():
    img_dir = paths['train_images']
    status = "Found" if os.path.isdir(img_dir) else "Missing"
    print(f"  {name:12s}  {status:7s}  {img_dir}")

---
## 5. Create Directories

Creates the checkpoint and log directories on Drive if they do not already exist.

**Before proceeding:** Confirm both directories show `exists: True`.

In [ ]:
import os
from config import CFG

os.makedirs(CFG.checkpoint_dir, exist_ok=True)
os.makedirs(CFG.log_dir, exist_ok=True)

print(f"Checkpoint dir: {CFG.checkpoint_dir}  (exists: {os.path.isdir(CFG.checkpoint_dir)})")
print(f"Log dir:        {CFG.log_dir}  (exists: {os.path.isdir(CFG.log_dir)})")

---
## 6. Preprocess Datasets

Generates Canny edge maps for all training and test masks. This only runs if edge map directories are empty or missing, so it is safe to re-run after a Colab reconnection without redoing work.

**Before proceeding:** Check the summary. Each dataset with data present should show a non-zero edge map count. Datasets marked as not found are skipped.

In [ ]:
import os
from config import CFG

needs_preprocessing = False
for name, paths in CFG.dataset_paths.items():
    edge_dir = paths['train_edges']
    masks_dir = paths['train_masks']
    if os.path.isdir(masks_dir):
        if not os.path.isdir(edge_dir) or len(os.listdir(edge_dir)) == 0:
            needs_preprocessing = True
            print(f"  {name}: edge maps missing -- will preprocess")
        else:
            print(f"  {name}: edge maps already exist ({len(os.listdir(edge_dir))} files) -- skipping")
    else:
        print(f"  {name}: dataset not found -- skipping")

if needs_preprocessing:
    print("\nRunning preprocessing...")
    !python preprocess.py all
else:
    print("\nAll edge maps already exist. No preprocessing needed.")

---
## 7. Smoke Test

Instantiates the MaS-TransUNet model, runs a single forward pass with dummy data, and prints output shapes and parameter count. This catches import errors, shape mismatches, or GPU memory issues before committing to a full training run.

**Before proceeding:** Verify all four output shapes are printed, the parameter count is ~775M, and there are no errors.

In [ ]:
import torch
from config import CFG
from models import build_model

model = build_model(CFG)

dummy_image = torch.randn(1, 3, 224, 224, device=CFG.device)
dummy_mask  = torch.zeros(1, 1, 224, 224, device=CFG.device)

with torch.no_grad():
    outputs = model(dummy_image, dummy_mask)

print("Output shapes:")
for key, val in outputs.items():
    print(f"  {key:10s}: {tuple(val.shape)}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params / 1e6:.2f}M")
print("Smoke test PASSED")

# Free memory
del model, dummy_image, dummy_mask, outputs
torch.cuda.empty_cache()

---
## 8. Auto-Select Datasets and Train

Automatically detects all datasets that are present and valid, then trains MaS-TransUNet on each one sequentially.

Training automatically resumes from the latest checkpoint if a Colab session disconnects. Checkpoints and previous-epoch masks are saved to Google Drive every 5 epochs.

**Before proceeding:** Ensure Cell 4 shows **Found** for at least one dataset.

In [ ]:
import os
from config import CFG

# Auto-detect datasets that actually exist and train them one by one
available_datasets = []
for name, paths in CFG.dataset_paths.items():
    train_images = paths["train_images"]
    train_masks = paths["train_masks"]
    test_images = paths["test_images"]
    test_masks = paths["test_masks"]

    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        available_datasets.append(name)

print("Datasets found:", available_datasets)

if not available_datasets:
    raise RuntimeError(
        "No valid datasets found. Run the dataset unzip/organize cell and ensure Cell 4 shows Found."
    )

for DATASET_NAME in available_datasets:
    print(f"\n=== Training: {DATASET_NAME} ===")
    !python train.py {DATASET_NAME}

print("\nAll available datasets have finished training.")

---
## 9. Evaluate

Runs evaluation on the test set using the best checkpoint and iterative FAM refinement (up to 10 iterations per sample). Prints a metrics table matching Tables II–VI in the paper, saves per-sample results to CSV, and writes qualitative side-by-side PNGs for the first 5 test images.

**Before proceeding:** Make sure training (Cell 8) has completed at least a few epochs and a `_best.pth` checkpoint exists.

In [ ]:
# Uses the same DATASET_NAME from Cell 8
!python evaluate.py {DATASET_NAME}

---
## 10. Training Order Note

The following cell explains the recommended order for training across all five datasets.

### Recommended Training Order

Train datasets in this order for fastest iteration and debugging:

| Order | Dataset      | Size     | Notes |
|:-----:|:-------------|:---------|:------|
| 1     | `tcga_lgg`   | Smallest | Start here to verify the full pipeline end-to-end with minimal time. Good for catching bugs early. |
| 2     | `dsb2018`    | Small    | Data Science Bowl 2018 nuclei segmentation. Quick to train, validates generalisation to cell-level tasks. |
| 3     | `kvasir_seg` | Medium   | Gastrointestinal polyp segmentation with more complex boundaries. Tests the EAM and edge loss. |
| 4     | `isic2018`   | Large    | Skin lesion segmentation with high variability. Longer training; monitor for overfitting. |
| 5     | `covid_ct`   | Largest  | COVID-19 CT scan segmentation. Most computationally expensive. Train last once everything else works. |

**To switch datasets:** Go back to **Cell 8**, change `DATASET_NAME`, and re-run Cells 8 and 9.

**Checkpointing:** All checkpoints are saved to Google Drive, so training resumes automatically after a Colab disconnection. Simply re-run from Cell 1.